# Efficient application of pipelines

A pipeline bundles preprocessing and the model so that everything is trained in the correct order without leaking information. It saves work by combining different models into one reusable object that will prevent us from messing up.

In [14]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [15]:
wine = load_wine()

X = wine.data
y = wine.target

Manual workflow:

In [16]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
# scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_pca, y_train)

# prediction
pred = rf.predict(X_test_pca)

print("Manual workflow accuracy:", accuracy_score(y_test, pred))

Manual workflow accuracy: 1.0


Pipeline workflow:

In [18]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=5)),
    ("rf", RandomForestClassifier(random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("Pipeline accuracy:", accuracy_score(y_test, y_pred))

Pipeline accuracy: 1.0


Pipeline in CV: Scaling, PCA and model training are repeated in every fold! (Likewise in GridSearchCV...)

In [21]:
scores_manual = cross_val_score(
    pipeline,
    X,
    y, 
    cv=5
)

print("Pipeline CV accuracy:", scores_manual.mean())

Pipeline CV accuracy: 0.9606349206349206
